In [0]:
%pip install arcgis

In [0]:
dbutils.library.restartPython()

In [0]:
%pip list | grep arcgis

In [0]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer
import pandas as pd

# Connect anonymously to ArcGIS Online (no credentials needed for public data)
gis = GIS()
print("Connected as:", gis)

In [0]:
# Example: Read a public ArcGIS Online layer into a Pandas DataFrame
layer_url = "https://services.arcgis.com/P3ePLMYs2RVChkJx/ArcGIS/rest/services/World_Cities/FeatureServer/0"
layer = FeatureLayer(layer_url)
sdf = layer.query().sdf # Returns a Spatially Enabled DataFrame
print(sdf.head())

# Pandas version of ArcGIS pointing to HFHI table

In [0]:
%skip
# ==============================================================================
# ArcGIS API for Python: Geocoding (Databricks)
# ==============================================================================

# --- 1. Imports ---
import pandas as pd                  # Standard library for tabular data manipulation
from arcgis.gis import GIS           # Manages connection and authentication to Esri
from arcgis.geocoding import geocode # The specific function to convert addresses to X/Y

# --- 2. Setup and Mock Data ---
# Initialize connection to ArcGIS Online. 
# Empty parentheses mean we connect anonymously (no account required for single requests).
gis = GIS()

# Read the Delta table from Unity Catalog using Spark
spark_df = spark.table("catalog_hfhi.gold.gold_constituents")

spark_df = spark_df.limit(3)

# Convert the Spark DataFrame to a Pandas DataFrame
# Note: Ensure driver node has enough memory if the table is very large.
df = spark_df.toPandas()

# --- 3. The Geocoding Function ---
def get_geocoding_details(row):
    """
    Takes a single row from the DataFrame, constructs a full address string,
    calls the ArcGIS geocoding service, and extracts the required metadata.
    """
    
    # Stitch the individual columns into a single comma-separated string.
    # Geocoders are much more accurate with a full "Single Line Address".
    address_str = f"{row['AddressLine1']}, {row['City']}, {row['StateProvince']} {row['PostalCode']}"
    
    try:
        # Call the API.
        # out_sr=4326 : Returns coordinates in standard Latitude/Longitude (WGS84).
        # out_fields="*" : Forces the API to return all metadata (Score, Status, etc.).
        results = geocode(address_str, out_sr=4326, out_fields="*")
        
        # Check if the API returned any candidate matches for the address
        if results:
            # Grab the best match (index 0 is always the highest confidence candidate)
            best_match = results[0]
            loc = best_match['location']       # Dictionary holding X and Y
            attr = best_match['attributes']    # Dictionary holding metadata
            
            # Return a Pandas Series containing exactly the data we need.
            # We use attr.get('Key', 'Default') so the code doesn't crash if a field is missing.
            return pd.Series([
                loc['y'],                     # Maps to: Latitude
                loc['x'],                     # Maps to: Longitude
                attr.get('Addr_type', ''),    # Maps to: GeocodingMatchMethod (e.g., PointAddress)
                attr.get('Score', 0),         # Maps to: GeocodingMatchScore (0-100)
                attr.get('Status', 'U'),      # Maps to: GeocodingMatchStatus (M=Matched, T=Tied, U=Unmatched)
                attr.get('Match_addr', '')    # Maps to: UniversalAddressID (Standardized Address)
            ])
        else:
            # Fallback if the server responds but finds absolutely zero matches
            return pd.Series([None, None, 'Unmatched', 0, 'U', None])
            
    except Exception as e:
        # Fallback if the network request fails entirely (e.g., timeout)
        print(f"Error geocoding {address_str}: {e}")
        return pd.Series([None, None, 'Error', 0, 'U', None])


# --- 4. Execution and Display ---
print("Sending addresses to ArcGIS World Geocoding Service...\n")

# Define the exact names for our new output columns.
# The order here must exactly match the order of the items returned in the pd.Series above.
output_columns = [
    'Latitude', 
    'Longitude', 
    'GeocodingMatchMethod', 
    'GeocodingMatchScore', 
    'GeocodingMatchStatus', 
    'UniversalAddressID'
]

# Apply the geocoding function to the DataFrame.
# axis=1 tells Pandas to process the data row-by-row and assign the output to our new columns.
df[output_columns] = df.apply(get_geocoding_details, axis=1)

# Render the final DataFrame as an interactive, formatted HTML table in Databricks
display(df[output_columns])

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:725)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:512)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:632)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$5(UsageLogging.scala:659)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:147)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:349)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:59)
	at com.databricks.logging.AttributionContext$.withValue(Att